In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Import your model environment
from utils.twostep_support import *
from models import *


In [2]:
import os
print(os.getcwd())

c:\Users\user\OneDrive\Desktop\data\McGill\Neur_503_computational-neuroscience\final_paper\aif-ddm-main


In [4]:
template_game = pd.read_csv(
    "two_step_task_datasets/synthetic_dataset_AI-DDM/00001_game.csv"
)

In [5]:
# -------------------------
# Simulation configuration
# -------------------------

model = "AI_ddm"
mtype = 3
drmtype = "linear"
learning = "PSM"

n_hc = 40
n_ps = 40
T = len(template_game)  # magic_carpet_2020 trial count

In [6]:
task_dict = {
    "type": "drift",
    "x": False,
    "r": True,
    "delta": 0.025,
    "bounds": [0.25, 0.75],
    "T": T
}

In [7]:
# -------------------------
# Healthy Controls
# -------------------------
hc_means = {
    "Synthetic_lr": 1.5,
    "Synthetic_vunsamp": 0.3,
    "Synthetic_vsamp": 0.6,
    "Synthetic_lam": 5,
    "Synthetic_kappa_a": 3.0,
    "Synthetic_prior_r": 0.5,
    "Synthetic_a_bs": 1.5,
    "Synthetic_ndt": 0.3,
    "Synthetic_v_stage_0": 2,
    "Synthetic_v_stage_1": 2
}

# -------------------------
# Psychosis Spectrum
# -------------------------
patient_means = {
    "Synthetic_lr": 0.8,          # slower updating
    "Synthetic_vunsamp": 0.6,     # stronger epistemic drive
    "Synthetic_vsamp": 0.3,
    "Synthetic_lam": 5,
    "Synthetic_kappa_a": 1.2,     # lower precision
    "Synthetic_prior_r": 0.7,     # stronger prior
    "Synthetic_a_bs": 1.5,
    "Synthetic_ndt": 0.3,
    "Synthetic_v_stage_0": 2,
    "Synthetic_v_stage_1": 2
}


In [8]:
def sample_group(means, n, label):
    rows = []
    
    for i in range(n):
        row = {}
        for param, mean in means.items():
            row[param] = np.random.normal(mean, abs(mean) * 0.1)
        row["Group"] = label
        rows.append(row)
    
    return pd.DataFrame(rows)

df_hc = sample_group(hc_means, n_hc, "HC")
df_ps = sample_group(patient_means, n_ps, "PS")

df_param_sets = pd.concat([df_hc, df_ps], ignore_index=True)
df_param_sets["ParticipantID"] = np.arange(1, len(df_param_sets)+1)

df_param_sets.head()


,Synthetic_lr,Synthetic_vunsamp,Synthetic_vsamp,Synthetic_lam,Synthetic_kappa_a,Synthetic_prior_r,Synthetic_a_bs,Synthetic_ndt,Synthetic_v_stage_0,Synthetic_v_stage_1,Group,ParticipantID
0,1.425323,0.242034,0.632262,4.195394,2.751455,0.462595,1.440989,0.250083,1.882619,1.826556,HC,1
1,1.527421,0.310358,0.460293,5.570790,3.129208,0.541522,1.185113,0.289669,1.900111,2.131328,HC,2
2,1.445729,0.292836,0.646296,5.100721,3.223836,0.443586,1.380465,0.286560,1.969898,2.091431,HC,3
3,1.585702,0.358870,0.648667,5.284348,2.949553,0.504063,1.539424,0.289494,1.825964,2.095480,HC,4
4,1.566371,0.312399,0.622186,6.022696,3.136564,0.475206,1.260419,0.301895,1.845953,2.401596,HC,5


In [9]:
df_param_sets.to_csv(
    "parameter_recovery/grouped_true_parameters_AI-DDM.csv",
    index=False
)

In [12]:
simulation_results = []

for i in tqdm(range(len(df_param_sets))):
    
    row = df_param_sets.iloc[i]
    
    model_dict = {
        "act": model,
        "learn": learning,
        "drmtype": drmtype,
        "learn_transitions": True,
        
        "lr": row["Synthetic_lr"],
        "vunsamp": row["Synthetic_vunsamp"],
        "vsamp": row["Synthetic_vsamp"],
        "vps": 0,

        "lam": row["Synthetic_lam"],
        "kappa_a": row["Synthetic_kappa_a"],
        "prior_r": row["Synthetic_prior_r"],
        "a_bs": row["Synthetic_a_bs"],
        "ndt": row["Synthetic_ndt"],
        "v_stage_0": row["Synthetic_v_stage_0"],
        "v_stage_1": row["Synthetic_v_stage_1"]
    }
    
    agent = learn_and_act(task=task_dict, model=model_dict)
    
    actions, observations, rts, pi, p_trans, p_r, Gs = agent.perform_task()
    
    df_participant = pd.DataFrame({
        "Synthetic_participant_ID": row["ParticipantID"],
        "Group": row["Group"],
        "trial": np.arange(T),
        "choice1": actions[:,0],
        "choice2": actions[:,1],
        "rt1": rts[:,0],
        "rt2": rts[:,1],
        "final_state": observations[:,0],
        "reward": observations[:,1]
    })
    
    simulation_results.append(df_participant)

df_simulated = pd.concat(simulation_results, ignore_index=True)

df_simulated.head()


100%|██████████| 80/80 [01:17<00:00,  1.03it/s]


,Synthetic_participant_ID,Group,trial,choice1,choice2,rt1,rt2,final_state,reward
0,1,HC,0,0,1,1.994801,0.857968,0,0
1,1,HC,1,0,0,0.313414,0.376008,0,1
2,1,HC,2,0,0,0.320936,1.739552,1,1
3,1,HC,3,0,0,0.310457,0.393371,0,0
4,1,HC,4,0,0,0.300369,0.343148,1,0


In [13]:
df_simulated_hc = df_simulated[df_simulated['Group'] == 'HC']
df_simulated_patient = df_simulated[df_simulated['Group'] == 'PS']

df_simulated_patient

,Synthetic_participant_ID,Group,trial,choice1,choice2,rt1,rt2,final_state,reward
8040,41,PS,0,0,0,0.559075,3.539874,0,0
8041,41,PS,1,0,1,0.703024,0.482832,0,1
8042,41,PS,2,0,1,0.495644,9.003469,1,0
8043,41,PS,3,0,1,0.416524,0.754507,0,0
8044,41,PS,4,0,0,0.841057,0.608432,1,0
...,...,...,...,...,...,...,...,...,...
16075,80,PS,196,0,1,0.618040,0.841919,0,1
16076,80,PS,197,0,0,0.640387,0.704991,1,0
16077,80,PS,198,0,1,0.446067,0.503073,0,1
16078,80,PS,199,0,1,0.510965,0.769155,1,1


In [14]:
import os
import shutil
import pandas as pd

project_root = r"C:\Users\user\OneDrive\Desktop\data\McGill\Neur_503_computational-neuroscience\final_paper\aif-ddm-main"

template_game_path = os.path.join(
    project_root,
    "two_step_task_datasets",
    "magic_carpet_2020_dataset",
    "choices",
    "03187_game.csv"
)

template_config_path = os.path.join(
    project_root,
    "two_step_task_datasets",
    "magic_carpet_2020_dataset",
    "choices",
    "03187_config.txt"
)

template_tutorial_path = os.path.join(
    project_root,
    "two_step_task_datasets",
    "magic_carpet_2020_dataset",
    "choices",
    "03187_tutorial.csv"
)

template_game = pd.read_csv(template_game_path)
template_tutorial = pd.read_csv(template_tutorial_path)

T = len(template_game)

save_root = os.path.join(
    project_root,
    "two_step_task_datasets",
    "synthetic_dataset_AI-DDM"
)

os.makedirs(save_root, exist_ok=True)

print("Template loaded.")


Template loaded.


In [15]:
for pid in df_simulated["Synthetic_participant_ID"].unique():
    
    df_p = df_simulated[df_simulated["Synthetic_participant_ID"] == pid]
    pid_str = f"{int(pid):05d}"

    # Start from template
    df_game = template_game.copy()

    # Overwrite only behavioural columns
    df_game["choice1"] = df_p["choice1"].values + 1
    df_game["choice2"] = df_p["choice2"].values + 1
    df_game["rt1"] = df_p["rt1"].values
    df_game["rt2"] = df_p["rt2"].values
    df_game["final_state"] = df_p["final_state"].values + 1
    df_game["reward"] = df_p["reward"].values

    # Save game file
    df_game.to_csv(
        os.path.join(save_root, f"{pid_str}_game.csv"),
        index=False
    )

    # Copy config
    shutil.copy(
        template_config_path,
        os.path.join(save_root, f"{pid_str}_config.txt")
    )

    # Copy tutorial
    template_tutorial.to_csv(
        os.path.join(save_root, f"{pid_str}_tutorial.csv"),
        index=False
    )

print("Synthetic dataset written in correct format.")


Synthetic dataset written in correct format.


In [16]:
df_simulated.to_csv("group_simulated_data/synthetic_two_step_AI_ddm_groups.csv", index=False)
df_simulated_hc.to_csv("group_simulated_data/synthetic_two_step_AI_ddm_HC.csv", index=False)
df_simulated_patient.to_csv("group_simulated_data/synthetic_two_step_AI_ddm_patient.csv", index=False)
df_param_sets.to_csv("group_simulated_data/synthetic_true_parameters_AI-ddm.csv", index=False)

print("Simulation complete.")


Simulation complete.
